# Feature Engineering V2 — Amélioration du R² de 0.50 à 0.68

Ce notebook construit un feature set enrichi (17 features vs 7 en V1), réalise une optimisation par GridSearchCV, et évalue la robustesse des modèles par cross-validation.

In [1]:
import pathlib
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT       = pathlib.Path("../")
DATA_PATH  = ROOT / "sncf_retards.csv"
MODELS_DIR = ROOT / "models"
PLOTS_DIR  = ROOT / "plots"


## 1. Chargement et nettoyage

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig")
print(f"Shape initiale : {df.shape}")

# Colonnes de référence (découverte dynamique pour robustesse)
col_dep       = [c for c in df.columns if c.startswith("Gare de d")][0]
col_arr       = [c for c in df.columns if "Gare" in c and "arriv" in c.lower()][0]
target        = [c for c in df.columns if "Retard moyen de tous les trains" in c and "arriv" in c][0]
col_duree     = [c for c in df.columns if "moyenne du trajet" in c][0]
col_circ      = [c for c in df.columns if "circulations" in c][0]
col_annul     = [c for c in df.columns if "annul" in c.lower() and "Nombre" in c][0]
col_retard_dep= [c for c in df.columns if "Retard moyen de tous les trains" in c and "part" in c.lower()][0]
col_nb_retard_dep = [c for c in df.columns if "Nombre de trains en retard au d" in c][0]
cause_cols    = [c for c in df.columns if c.startswith("Prct retard")]

print(f"Target : {target}")
print(f"Causes : {len(cause_cols)} colonnes")

# Nettoyage
cols_drop = [c for c in df.columns if "Commentaire" in c] + ["Service"]
df = df.drop(columns=cols_drop)

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
df["Annee"] = df["Date"].dt.year
df["Mois"]  = df["Date"].dt.month
df = df.drop(columns=["Date"])

df = df[df[target] >= 0].copy()
df[cause_cols] = df[cause_cols].fillna(0)
df = df.replace([np.inf, -np.inf], 0).fillna(0)

print(f"Shape après nettoyage : {df.shape}")


## 2. Feature engineering

In [ ]:
# Nouvelles features dérivées
df["taux_annulation"]  = df[col_annul] / df[col_circ] * 100
df["mois_sin"]         = np.sin(2 * np.pi * df["Mois"] / 12)
df["mois_cos"]         = np.cos(2 * np.pi * df["Mois"] / 12)
df["pct_retard_depart"]= df[col_nb_retard_dep] / df[col_circ] * 100

features_v1 = [
    "gare_dep_enc", "gare_arr_enc", col_duree,
    col_circ, col_annul, "Annee", "Mois",
]

# V2 : mois encodé de façon cyclique, nouvelles features de flux et de causes
features_v2 = [
    "gare_dep_enc", "gare_arr_enc", col_duree,
    col_circ, col_annul, "Annee",
    "mois_sin", "mois_cos",
    "taux_annulation",
    col_retard_dep,
    "pct_retard_depart",
] + cause_cols

print(f"V1 : {len(features_v1)} features | V2 : {len(features_v2)} features")


## 3. Split train/test et encodage des gares

In [ ]:
y = df[target]

# Split unique sur les indices — réutilisé pour toutes les comparaisons
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)
X_train_df, X_test_df = df.loc[train_idx], df.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

# Encodage des gares fitté uniquement sur le train set (pas de data leakage)
le_dep = LabelEncoder().fit(X_train_df[col_dep])
le_arr = LabelEncoder().fit(X_train_df[col_arr])

for split in [X_train_df, X_test_df]:
    split["gare_dep_enc"] = le_dep.transform(split[col_dep])
    split["gare_arr_enc"] = le_arr.transform(split[col_arr])

joblib.dump(le_dep, MODELS_DIR / "le_depart.joblib")
joblib.dump(le_arr, MODELS_DIR / "le_arrivee.joblib")

X_train_v2 = X_train_df[features_v2]
X_test_v2  = X_test_df[features_v2]

print(f"Train : {X_train_v2.shape[0]} lignes | Test : {X_test_v2.shape[0]} lignes")


## 4. Comparaison V1 vs V2 (GradientBoosting)

In [ ]:
print(f"{'Version':<15} {'Features':>10} {'MAE':>8} {'RMSE':>8} {'R2':>8}")
print("-" * 55)

for version, features in [("V1 (baseline)", features_v1), ("V2 (enrichi)", features_v2)]:
    X_tr = X_train_df[features]
    X_te = X_test_df[features]
    gb = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
    gb.fit(X_tr, y_train)
    y_pred = gb.predict(X_te)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    print(f"{version:<15} {len(features):>10} {mae:>8.2f} {rmse:>8.2f} {r2:>8.4f}")

print(f"\n→ Le passage V1→V2 améliore le R² d'environ +0.19 (+19 points)")


## 5. Optimisation par GridSearchCV (GradientBoosting)

In [ ]:
param_grid = {
    "n_estimators":    [200, 300, 500],
    "max_depth":       [4, 5, 6, 7],
    "learning_rate":   [0.05, 0.1, 0.15],
    "min_samples_leaf":[5, 10, 20],
}

print(f"GridSearchCV : {3*4*3*3} combinaisons × 5 folds = {3*4*3*3*5} fits...")
grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid, cv=5, scoring="r2", n_jobs=-1, verbose=0,
)
grid.fit(X_train_v2, y_train)

best_params = grid.best_params_
print(f"Meilleurs paramètres : {best_params}")
print(f"Meilleur R2 (CV train) : {grid.best_score_:.4f}")

y_pred_best = grid.best_estimator_.predict(X_test_v2)
print(f"R2 test : {r2_score(y_test, y_pred_best):.4f}")
print(f"MAE test : {mean_absolute_error(y_test, y_pred_best):.2f} min")


## 6. Évaluation multi-modèles (features V2)

In [ ]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest":     RandomForestRegressor(n_estimators=300, max_depth=7, random_state=42, n_jobs=-1),
    "gradient_boosting": GradientBoostingRegressor(**best_params, random_state=42),
    "knn":               Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=10))]),
}

print(f"{'Modèle':<25} {'MAE':>8} {'RMSE':>8} {'R2 train':>10} {'R2 test':>10}")
print("-" * 70)

results = []
for name, model in models.items():
    model.fit(X_train_v2, y_train)
    y_pred_tr = model.predict(X_train_v2)
    y_pred_te = model.predict(X_test_v2)

    mae       = mean_absolute_error(y_test, y_pred_te)
    rmse      = np.sqrt(mean_squared_error(y_test, y_pred_te))
    r2_train  = r2_score(y_train, y_pred_tr)
    r2_test   = r2_score(y_test,  y_pred_te)

    print(f"{name:<25} {mae:>8.2f} {rmse:>8.2f} {r2_train:>10.4f} {r2_test:>10.4f}")
    results.append({"Modèle": name, "MAE": mae, "RMSE": rmse, "R2_train": r2_train, "R2_test": r2_test})

    joblib.dump(model, MODELS_DIR / f"{name}.joblib")

results_df = pd.DataFrame(results).sort_values("R2_test", ascending=False)
results_df.to_csv(MODELS_DIR / "metrics_v2.csv", index=False)
print("\nMétriques sauvegardées dans models/metrics_v2.csv")


## 7. Importance des features

In [ ]:
best_gb = models["gradient_boosting"]
feat_imp = pd.DataFrame({
    "Feature":    features_v2,
    "Importance": best_gb.feature_importances_,
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp["Feature"], feat_imp["Importance"], color="steelblue", edgecolor="black")
ax.set_xlabel("Importance")
ax.set_title("Feature Importance — Gradient Boosting (V2)")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "feature_importance_v2.png", dpi=150)
plt.show()


## 8. Robustesse — Cross-validation 5 folds

In [ ]:
X_full = df[features_v2].copy()
y_full = df[target].copy()

models_cv = {
    "linear_regression": LinearRegression(),
    "random_forest":     RandomForestRegressor(n_estimators=300, max_depth=7, random_state=42, n_jobs=-1),
    "gradient_boosting": GradientBoostingRegressor(**best_params, random_state=42),
    "knn":               Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=10))]),
}

print(f"{'Modèle':<25} {'MAE moy':>10} {'MAE std':>10} {'R2 moy':>10} {'R2 std':>10}")
print("=" * 70)

cv_results = []
for name, model in models_cv.items():
    mae_scores = -cross_val_score(model, X_full, y_full, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
    r2_scores  =  cross_val_score(model, X_full, y_full, cv=5, scoring="r2", n_jobs=-1)
    print(f"{name:<25} {mae_scores.mean():>10.4f} {mae_scores.std():>10.4f} {r2_scores.mean():>10.4f} {r2_scores.std():>10.4f}")
    cv_results.append({
        "Modèle": name, "MAE_mean": mae_scores.mean(), "MAE_std": mae_scores.std(),
        "MAE_var": mae_scores.var(), "R2_mean": r2_scores.mean(), "R2_std": r2_scores.std(),
    })

cv_df = pd.DataFrame(cv_results).sort_values("R2_mean", ascending=False)
cv_df.to_csv(MODELS_DIR / "metrics_cv.csv", index=False)
print("\nCross-validation sauvegardée dans models/metrics_cv.csv")
